In [ ]:
import numpy as np

# import json

# Load the NPZ file containing your warp and angle maps
warp_data = np.load("/home/markus/git/inv3d-generator/tmp_dir_sample_001/warped_UV.npz")
bm_map = warp_data["warped_UV"]  # Assuming this contains the warping coordinates
# angle_map = warp_data['angle_map']  # Warped angle information

# # Load your JSON with the original bounding boxes
# with open('/home/markus/git/inv3d-generator/tmp_dir_sample_001/annotation.json', 'r') as f:
#     bbox_data = json.load(f)

# bbox = bbox_data["supplier_name"]

In [ ]:
import cv2
import numpy as np
from PIL import Image

# Load and prepare the template image
# template_padded = Image.new("RGB", (512, 512), "black")
# # template_flat = Image.open("/home/markus/git/inv3d-generator/tmp_dir_sample_001/flat_template.png")
# template_flat = Image.open("/home/markus/git/inv3d-generator/tmp_dir_sample_001/warped_document.png")
# template_flat.thumbnail((512, 512))
# template_padded.paste(template_flat, (0, 0))


img_warped = Image.open(
    "/home/markus/git/inv3d-generator/tmp_dir_sample_001/warped_document.png"
)
img_warped_resized = img_warped.resize((512, 512))

# Convert to OpenCV format
original_img = cv2.cvtColor(np.array(img_warped_resized), cv2.COLOR_RGB2BGR)

# Load backward map
data = np.load("/home/markus/git/inv3d-generator/tmp_dir_sample_001/warped_BM.npz")
backward_map = data["warped_BM"].astype(np.float32)  # Ensure float32

# Get dimensions
h, w = backward_map.shape[:2]

# Scale the backward map to match image dimensions
backward_map[..., 0] *= w  # Scale x-coordinates
backward_map[..., 1] *= h  # Scale y-coordinates

# Swap coordinates if needed (OpenCV expects x, y format)
backward_map_swapped = np.zeros_like(backward_map)
backward_map_swapped[..., 0] = backward_map[..., 1]  # y -> x
backward_map_swapped[..., 1] = backward_map[..., 0]  # x -> y

# Perform remapping
dewarped = cv2.remap(original_img, backward_map_swapped, None, cv2.INTER_CUBIC)

# Convert back to PIL for display
dewarped_rgb = cv2.cvtColor(dewarped, cv2.COLOR_BGR2RGB)
img_dewarped = Image.fromarray(dewarped_rgb)

# Display results
display(img_warped_resized)
display(img_dewarped)

In [ ]:
from PIL import Image, ImageDraw

# (2024) no need RGBA if you don't use `composite()` (besides JPG can't write RGBA)
bbox = ((259, 103), (259 + 221, 103 + 71))  # compare gimp

draw = ImageDraw.Draw(img_dewarped)
draw.rectangle(bbox, outline="red")

img_dewarped

In [ ]:
(*bbox[0], *bbox[1])

In [ ]:
import numpy as np


def map_bounding_box_to_original(x1_w, y1_w, x2_w, y2_w, backward_map):
    """
    Maps the bounding box (x1', y1', x2', y2') from the warped image back to
    the original image using backward_map.

    Arguments:
    - x1_w, y1_w, x2_w, y2_w: Bounding box in the warped image.
    - backward_map: The mapping array (h, w, 2) from warped to original.

    Returns:
    - (x1_o, y1_o, x2_o, y2_o): Bounding box coordinates in the original image.
    """

    h, w = backward_map_swapped.shape[:2]  # Get dimensions of the warped image

    # Ensure bounding box is within image bounds
    x1_w, x2_w = max(0, x1_w), min(w - 1, x2_w)
    y1_w, y2_w = max(0, y1_w), min(h - 1, y2_w)

    # # Map each bounding box corner back to the original image
    # x1_o, y1_o = backward_map_swapped[y1_w, x1_w]  # Top-left
    # x2_o, y1_o = backward_map_swapped[y1_w, x2_w]  # Top-right
    # x1_o, y2_o = backward_map_swapped[y2_w, x1_w]  # Bottom-left
    # x2_o, y2_o = backward_map_swapped[y2_w, x2_w]  # Bottom-right

    # # Return transformed bounding box in the original image
    # return np.array((x1_o, y1_o, x2_o, y2_o)).astype(int)
    # Get the transformed coordinates of all 4 corners
    top_left = backward_map_swapped[y1_w, x1_w]  # (x, y)
    top_right = backward_map_swapped[y1_w, x2_w]
    bottom_left = backward_map_swapped[y2_w, x1_w]
    bottom_right = backward_map_swapped[y2_w, x2_w]

    # Extract x and y coordinates
    x_coords = [top_left[0], top_right[0], bottom_left[0], bottom_right[0]]
    y_coords = [top_left[1], top_right[1], bottom_left[1], bottom_right[1]]
    # Extract x and y coordinates
    x_coords = [top_left[0], top_right[0], bottom_left[0], bottom_right[0]]
    y_coords = [top_left[1], top_right[1], bottom_left[1], bottom_right[1]]

    # Compute the enclosing bounding box
    x_min, x_max = min(x_coords), max(x_coords)
    y_min, y_max = min(y_coords), max(y_coords)

    return int(x_min), int(y_min), int(x_max), int(y_max)


# Example usage
# x1_w, y1_w, x2_w, y2_w) = bbox  # Bounding box in warped image
original_bbox = map_bounding_box_to_original(
    bbox[0][0], bbox[0][1], bbox[1][0], bbox[1][1], backward_map
)

print("Bounding box in original image:", original_bbox)

In [ ]:
draw = ImageDraw.Draw(img_warped_resized)
draw.rectangle(((296, 166), (444, 236)), outline="red")
img_warped_resized